# Double Machine Learning with 401(k) Data: From Eligibility Effects to Complier Analysis

This notebook is a runnable companion to the blog post [*Double Machine Learning with 401(k) Data*](https://carlos-mendez.org/post/python_doubleml_pension/).

Does having access to a 401(k) plan actually cause households to save more, or do households with 401(k) access simply have higher incomes and save more regardless? A naive comparison shows that 401(k)-eligible households have \$19,559 more in net financial assets than ineligible ones. But this number is almost certainly inflated by *confounders* — variables like income and education that affect both 401(k) access and savings.

**Double Machine Learning (DML)** solves this problem by using flexible ML models to partial out the confounding variation, then estimating the causal effect on the cleaned residuals. In this notebook we apply three DML models — PLR, IRM, and IIVM — to the classic 401(k) pension dataset from the 1991 Survey of Income and Program Participation (SIPP).

> **How to use this notebook:** click `Runtime → Run all` (or press `Ctrl+F9`) to execute every cell. The full pipeline takes about 2–3 minutes on a free Colab CPU runtime.

**Learning objectives:**

- Understand three DoubleML models (PLR, IRM, IIVM) and when to use each
- Distinguish between the Average Treatment Effect (ATE) and the Local Average Treatment Effect (LATE)
- Apply four different ML learners as nuisance estimators and assess robustness
- Interpret the gap between naive and DML estimates as evidence of confounding bias
- Use instrumental variables within the DML framework to handle endogenous treatment

## Setup and imports

Install the required packages (DoubleML and XGBoost):

In [ ]:
!pip install doubleml xgboost --quiet

Import all necessary libraries and set configuration variables. We use `RANDOM_SEED = 42` throughout for reproducibility.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import doubleml as dml
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LassoCV, LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.pipeline import make_pipeline
from xgboost import XGBClassifier, XGBRegressor
from doubleml.datasets import fetch_401K
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Configuration
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Site color palette
STEEL_BLUE = "#6a9bcc"
WARM_ORANGE = "#d97757"
NEAR_BLACK = "#141413"
TEAL = "#00d4c8"
GRAY = "#999999"

# Method color scheme
COLOR_PLR = STEEL_BLUE
COLOR_IRM = WARM_ORANGE
COLOR_IIVM = TEAL
COLOR_NAIVE = GRAY

# Variables
OUTCOME = "net_tfa"
OUTCOME_LABEL = "Net Total Financial Assets ($)"
TREATMENT_ELIG = "e401"
TREATMENT_PART = "p401"
FEATURES_BASE = ["age", "inc", "educ", "fsize", "marr",
                 "twoearn", "db", "pira", "hown"]

# Regularization grid for LogisticRegressionCV
Cs = 0.0001 * np.logspace(0, 4, 10)

## Data loading: the 401(k) pension dataset

The dataset comes from the 1991 Survey of Income and Program Participation (SIPP), a nationally representative survey of U.S. households. It contains 9,915 observations with information on 401(k) eligibility, participation, financial assets, and demographic characteristics.

The distinction between `e401` (eligibility) and `p401` (participation) is crucial. Eligibility is determined largely by the *employer* — whether the company offers a 401(k) plan. Participation is a *household decision* — whether the eligible household actually enrolls. This matters because eligibility is plausibly unrelated to individual savings behavior (after controlling for income and other characteristics), while participation is a choice driven partly by unobservable traits like financial discipline.

In [ ]:
data = fetch_401K(return_type="DataFrame")
print(f"Dataset shape: {data.shape}")
print(f"\nVariable descriptions:")
print(f"  net_tfa  : Net total financial assets (outcome)")
print(f"  e401     : 401(k) eligibility (treatment / instrument)")
print(f"  p401     : 401(k) participation (endogenous treatment)")
print(f"  age      : Age of household head")
print(f"  inc      : Income")
print(f"  educ     : Education level")
print(f"  fsize    : Family size")
print(f"  marr     : Marital status")
print(f"  twoearn  : Two-earner household")
print(f"  db       : Defined benefit pension")
print(f"  pira     : IRA participation")
print(f"  hown     : Home ownership")

print(f"\nOutcome summary (net_tfa):")
print(data[OUTCOME].describe().round(2))

print(f"\nTreatment rates:")
print(f"  Eligible (e401=1): {data[TREATMENT_ELIG].sum()} / {len(data)} "
      f"({data[TREATMENT_ELIG].mean():.1%})")
print(f"  Participating (p401=1): {data[TREATMENT_PART].sum()} / {len(data)} "
      f"({data[TREATMENT_PART].mean():.1%})")

The dataset contains 9,915 U.S. households. About 37% are eligible for a 401(k) plan and 26% actually participate, meaning roughly 70% of eligible households choose to enroll. Net total financial assets (`net_tfa`) are highly skewed: the median is just \$1,499, while the mean is \$18,052.

## Exploratory data analysis

Before estimating causal effects, let us visualize the data to understand the outcome distribution and the confounding structure.

### Outcome distribution by eligibility

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left panel: histograms
for val, label, color in [(1, "Eligible (e401=1)", STEEL_BLUE),
                           (0, "Not eligible (e401=0)", WARM_ORANGE)]:
    subset = data[data[TREATMENT_ELIG] == val][OUTCOME]
    axes[0].hist(subset, bins=50, alpha=0.6, label=label, color=color,
                 edgecolor="white", linewidth=0.5)
axes[0].set_xlabel(OUTCOME_LABEL)
axes[0].set_ylabel("Frequency")
axes[0].set_title("Distribution of Net Financial Assets\nby 401(k) Eligibility")
axes[0].legend(frameon=False)
axes[0].set_xlim(-50000, 200000)

# Right panel: box plots
bp_data = [data[data[TREATMENT_ELIG] == 0][OUTCOME].values,
           data[data[TREATMENT_ELIG] == 1][OUTCOME].values]
bp = axes[1].boxplot(bp_data, tick_labels=["Not Eligible", "Eligible"],
                     patch_artist=True, widths=0.5,
                     medianprops=dict(color=NEAR_BLACK, linewidth=2))
bp["boxes"][0].set_facecolor(WARM_ORANGE)
bp["boxes"][0].set_alpha(0.6)
bp["boxes"][1].set_facecolor(STEEL_BLUE)
bp["boxes"][1].set_alpha(0.6)
axes[1].set_ylabel(OUTCOME_LABEL)
axes[1].set_title("Net Financial Assets\nby 401(k) Eligibility")
axes[1].set_ylim(-50000, 200000)

plt.tight_layout()
plt.show()

Eligible households clearly have higher and more dispersed financial assets. But is this gap driven by 401(k) access itself, or by underlying differences between the two groups?

### Income confounding

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: income histograms by eligibility
for val, label, color in [(1, "Eligible (e401=1)", STEEL_BLUE),
                           (0, "Not eligible (e401=0)", WARM_ORANGE)]:
    subset = data[data[TREATMENT_ELIG] == val]["inc"]
    axes[0].hist(subset, bins=50, alpha=0.6, label=label, color=color,
                 edgecolor="white", linewidth=0.5)
axes[0].set_xlabel("Income ($)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Income Distribution by 401(k) Eligibility\n(Key Confounder)")
axes[0].legend(frameon=False)

# Right: income vs net_tfa colored by eligibility
sample = data.sample(n=min(2000, len(data)), random_state=RANDOM_SEED)
for val, label, color in [(0, "Not eligible", WARM_ORANGE),
                           (1, "Eligible", STEEL_BLUE)]:
    subset = sample[sample[TREATMENT_ELIG] == val]
    axes[1].scatter(subset["inc"], subset[OUTCOME], alpha=0.3,
                    s=15, color=color, label=label)
axes[1].set_xlabel("Income ($)")
axes[1].set_ylabel(OUTCOME_LABEL)
axes[1].set_title("Income vs. Net Financial Assets\n(Confounding Visualized)")
axes[1].legend(frameon=False)
axes[1].set_ylim(-50000, 200000)

plt.tight_layout()
plt.show()

The left panel reveals the confounding structure: eligible households earn substantially more on average (\$46,862 vs. \$31,494 for ineligible households), a gap of over \$15,000. The scatter plot confirms that income drives both eligibility and assets — eligible households (blue) cluster in the upper-right region of higher income and higher wealth.

### Summary statistics

In [ ]:
eda_summary = data.groupby(TREATMENT_ELIG).agg(
    n=("net_tfa", "size"),
    mean_net_tfa=("net_tfa", "mean"),
    median_net_tfa=("net_tfa", "median"),
    std_net_tfa=("net_tfa", "std"),
    mean_income=("inc", "mean"),
    mean_age=("age", "mean"),
    mean_educ=("educ", "mean"),
).round(2)
eda_summary.index = eda_summary.index.map({0: "Not Eligible", 1: "Eligible"})
eda_summary.index.name = "Eligibility"
print("Summary by eligibility status:")
print(eda_summary.to_string())

The summary table quantifies the selection problem: eligible households differ from ineligible ones on every observable dimension — they have higher income, more education, and are slightly older. Any comparison that does not account for these differences will overstate the causal effect.

## The naive benchmark

Before applying DML, let us compute the naive difference-in-means:

$$\hat{\Delta}_{naive} = \bar{Y}_{e401=1} - \bar{Y}_{e401=0}$$

This estimator lumps together the genuine causal effect with all pre-existing differences:

$$\hat{\Delta}_{naive} = \underbrace{\theta_0}_{\text{causal effect}} + \underbrace{\text{bias from confounders}}_{\text{income, education, ...}}$$

As we will see, the naive \$19,559 decomposes into roughly \$8,730 of genuine causal effect and \$10,829 of confounding bias — more than half the raw gap is an illusion.

In [ ]:
mean_elig = data[data[TREATMENT_ELIG] == 1][OUTCOME].mean()
mean_noelig = data[data[TREATMENT_ELIG] == 0][OUTCOME].mean()
naive_elig = mean_elig - mean_noelig

mean_part = data[data[TREATMENT_PART] == 1][OUTCOME].mean()
mean_nopart = data[data[TREATMENT_PART] == 0][OUTCOME].mean()
naive_part = mean_part - mean_nopart

print(f"Naive difference (eligibility): ${naive_elig:,.2f}")
print(f"  Eligible mean:     ${mean_elig:,.2f}")
print(f"  Not eligible mean: ${mean_noelig:,.2f}")
print(f"\nNaive difference (participation): ${naive_part:,.2f}")
print(f"  Participating mean:     ${mean_part:,.2f}")
print(f"  Not participating mean: ${mean_nopart:,.2f}")

The naive comparison suggests that 401(k) eligibility is associated with \$19,559 more in financial assets, and participation with \$27,372 more. These numbers are informative as benchmarks, but they are almost certainly biased upward.

## Data preparation for DoubleML

We prepare two covariate specifications. The **base** specification uses 9 raw features. The **flexible** specification adds quadratic terms for continuous variables (age, income, education, family size), giving the Lasso learner a richer set of features.

In [ ]:
# Base specification: 9 raw features
data_dml_base = dml.DoubleMLData(data,
                                 y_col=OUTCOME,
                                 d_cols=TREATMENT_ELIG,
                                 x_cols=FEATURES_BASE)
print(f"Base specification: {len(FEATURES_BASE)} features")
print(f"  Features: {FEATURES_BASE}")

# Flexible specification: polynomial features for Lasso
features_flex = data.copy()[["marr", "twoearn", "db", "pira", "hown"]]
poly_dict = {"age": 2, "inc": 2, "educ": 2, "fsize": 2}
for key, degree in poly_dict.items():
    poly = PolynomialFeatures(degree, include_bias=False)
    data_transf = poly.fit_transform(data[[key]])
    x_cols = poly.get_feature_names_out([key])
    data_transf = pd.DataFrame(data_transf, columns=x_cols)
    features_flex = pd.concat((features_flex, data_transf),
                              axis=1, sort=False)

model_data_elig = pd.concat(
    (data.copy()[[OUTCOME, TREATMENT_ELIG]], features_flex.copy()),
    axis=1, sort=False
)
data_dml_flex = dml.DoubleMLData(model_data_elig,
                                 y_col=OUTCOME,
                                 d_cols=TREATMENT_ELIG)
print(f"\nFlexible specification: {features_flex.shape[1]} features")
print(f"  Includes quadratic terms for: {list(poly_dict.keys())}")

# IV data: treatment = p401 (participation), instrument = e401 (eligibility)
data_dml_base_iv = dml.DoubleMLData(data,
                                    y_col=OUTCOME,
                                    d_cols=TREATMENT_PART,
                                    z_cols=TREATMENT_ELIG,
                                    x_cols=FEATURES_BASE)

model_data_iv = pd.concat(
    (data.copy()[[OUTCOME, TREATMENT_ELIG, TREATMENT_PART]],
     features_flex.copy()),
    axis=1, sort=False
)
data_dml_iv_flex = dml.DoubleMLData(model_data_iv,
                                    y_col=OUTCOME,
                                    d_cols=TREATMENT_PART,
                                    z_cols=TREATMENT_ELIG)
print(f"\nIV specification: treatment=p401, instrument=e401")

## ML learner definitions

We define four pairs of ML learners (regression + classification) that will be used across all three DML models. Each pair consists of a regressor (for outcome prediction) and a classifier (for treatment/propensity prediction).

In [ ]:
def get_learners():
    """Return dictionaries of regression and classification learners."""
    regressors = {
        "Lasso": make_pipeline(StandardScaler(),
                               LassoCV(cv=5, max_iter=10000)),
        "Random Forest": RandomForestRegressor(
            n_estimators=500, max_depth=7, max_features=3,
            min_samples_leaf=3, random_state=RANDOM_SEED),
        "Decision Tree": DecisionTreeRegressor(
            max_depth=30, ccp_alpha=0.0047,
            min_samples_split=203, min_samples_leaf=67,
            random_state=RANDOM_SEED),
        "XGBoost": XGBRegressor(
            n_jobs=1, objective="reg:squarederror",
            eta=0.1, n_estimators=35, random_state=RANDOM_SEED),
    }
    classifiers = {
        "Lasso": make_pipeline(StandardScaler(),
                               LogisticRegressionCV(
                                   cv=5, penalty="l1", solver="liblinear",
                                   Cs=Cs, max_iter=1000)),
        "Random Forest": RandomForestClassifier(
            n_estimators=500, max_depth=5, max_features=4,
            min_samples_leaf=7, random_state=RANDOM_SEED),
        "Decision Tree": DecisionTreeClassifier(
            max_depth=30, ccp_alpha=0.0042,
            min_samples_split=104, min_samples_leaf=34,
            random_state=RANDOM_SEED),
        "XGBoost": XGBClassifier(
            n_jobs=1, objective="binary:logistic",
            eval_metric="logloss",
            eta=0.1, n_estimators=34, random_state=RANDOM_SEED),
    }
    return regressors, classifiers

print("Learners defined: Lasso, Random Forest, Decision Tree, XGBoost")

## Model 1: Partially Linear Regression (PLR)

The PLR model is the workhorse of DML. It assumes a constant treatment effect $\theta_0$ while allowing the confounding structure to be arbitrarily complex:

$$Y = \theta_0 \, D + g_0(X) + \varepsilon, \quad E[\varepsilon \mid D, X] = 0$$

$$D = m_0(X) + V, \quad E[V \mid X] = 0$$

**How partialling out works — a three-step recipe:**

1. **Predict the outcome from covariates.** Train ML to predict net financial assets ($Y$) using only covariates ($X$). The residual $\tilde{Y} = Y - \hat{g}_0(X)$ is the part of savings unexplained by demographics.

2. **Predict the treatment from covariates.** Train ML to predict eligibility ($D$) from the same covariates. The residual $\tilde{D} = D - \hat{m}_0(X)$ is "surprise eligibility."

3. **Regress outcome residuals on treatment residuals.** The slope $\hat{\theta}_0$ is our causal estimate — cleaned of confounding.

Think of it like noise-canceling headphones: the ML models learn the "noise" pattern from confounders, subtract it, and what remains is the clean causal signal.

DML also uses **cross-fitting** (3-fold): each fold's residuals come from a model trained on the other two folds, preventing overfitting bias.

In [ ]:
print("Estimand: Average Treatment Effect (ATE) of 401(k) eligibility")
print("Treatment: e401 (eligibility)")
print("PLR assumes: Y = theta * D + g(X) + epsilon\n")

plr_results = []
regressors, classifiers = get_learners()

for name in regressors:
    np.random.seed(RANDOM_SEED)
    dml_data = data_dml_flex if name == "Lasso" else data_dml_base

    model = dml.DoubleMLPLR(dml_data,
                            ml_l=regressors[name],
                            ml_m=classifiers[name],
                            n_folds=3)
    model.fit(store_predictions=True)

    coef = model.coef[0]
    se = model.se[0]
    ci = model.confint(level=0.95).values[0]

    plr_results.append({
        "Model": "PLR",
        "Learner": name,
        "Coefficient": round(coef, 2),
        "Std_Error": round(se, 2),
        "CI_Lower": round(ci[0], 2),
        "CI_Upper": round(ci[1], 2),
        "Estimand": "ATE",
    })
    print(f"PLR-{name}: coef={coef:,.2f}, SE={se:,.2f}, "
          f"95% CI=[{ci[0]:,.2f}, {ci[1]:,.2f}]")

plr_df = pd.DataFrame(plr_results)
print("\nPLR results:")
print(plr_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
y_pos = range(len(plr_df))
ax.barh(y_pos, plr_df["Coefficient"], xerr=[
    plr_df["Coefficient"] - plr_df["CI_Lower"],
    plr_df["CI_Upper"] - plr_df["Coefficient"]],
    color=COLOR_PLR, alpha=0.7, edgecolor="white", capsize=4)
ax.axvline(x=naive_elig, color=COLOR_NAIVE, linestyle="--", linewidth=1.5,
           label=f"Naive difference (${naive_elig:,.0f})")
ax.set_yticks(y_pos)
ax.set_yticklabels(plr_df["Learner"])
ax.set_xlabel("Estimated Effect on Net Financial Assets ($)")
ax.set_title("PLR Estimates: Effect of 401(k) Eligibility (ATE)\nAcross ML Learners")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

After controlling for confounders, the PLR model estimates the ATE at \$7,823 to \$9,371 across the four learners, with a mean of \$8,730. Compare this to the naive estimate of \$19,559: DML reveals that roughly \$10,829 (55%) of the raw gap was confounding bias. The narrow range across learners (\$1,548) demonstrates robustness to the choice of ML algorithm.

## Model 2: Interactive Regression Model (IRM)

While PLR uses a *partialling-out* strategy, IRM takes a different approach rooted in the *potential outcomes framework*. It combines an outcome model $g_0(D, X)$ and a propensity score model $m_0(X) = P(D=1 \mid X)$ into a **doubly robust** (AIPW) estimator:

$$\theta_0 = E\left[g_0(1, X) - g_0(0, X) + \frac{D \, (Y - g_0(1, X))}{m_0(X)} - \frac{(1-D) \, (Y - g_0(0, X))}{1-m_0(X)}\right]$$

The propensity score $m_0(X)$ is simply the predicted probability that a household is eligible, based on its characteristics. For a high-income, well-educated household, this might be 0.70; for a low-income, young household, it might be 0.15.

"Doubly robust" means the estimator is consistent if *either* the outcome model or the propensity score model is correctly specified. Think of it as a safety net: if one model stumbles, the other catches it.

Running both PLR and IRM is like getting a second opinion from a different doctor using a different diagnostic approach.

In [ ]:
print("Estimand: Average Treatment Effect (ATE) of 401(k) eligibility")
print("IRM uses doubly robust / AIPW estimation with propensity scores.")
print("Uses trimming_threshold=0.01 to handle extreme propensity scores.\n")

irm_results = []
regressors, classifiers = get_learners()

# IRM-specific nuisance parameter tuning (from DoubleML docs)
irm_nuisance_params = {
    "Random Forest": {
        "ml_g0": {"max_depth": 6, "max_features": 4, "min_samples_leaf": 7},
        "ml_g1": {"max_depth": 6, "max_features": 3, "min_samples_leaf": 5},
        "ml_m": {"max_depth": 6, "max_features": 3, "min_samples_leaf": 6},
    },
    "Decision Tree": {
        "ml_g0": {"ccp_alpha": 0.0016, "min_samples_split": 74,
                  "min_samples_leaf": 24},
        "ml_g1": {"ccp_alpha": 0.0018, "min_samples_split": 70,
                  "min_samples_leaf": 23},
        "ml_m": {"ccp_alpha": 0.0028, "min_samples_split": 167,
                 "min_samples_leaf": 55},
    },
    "XGBoost": {
        "ml_g0": {"eta": 0.1, "n_estimators": 8},
        "ml_g1": {"eta": 0.1, "n_estimators": 29},
        "ml_m": {"eta": 0.1, "n_estimators": 23},
    },
}

for name in regressors:
    np.random.seed(RANDOM_SEED)
    if name == "Lasso":
        lasso_reg = make_pipeline(StandardScaler(),
                                  LassoCV(cv=5, max_iter=20000))
        model = dml.DoubleMLIRM(data_dml_flex,
                                ml_g=lasso_reg,
                                ml_m=classifiers[name],
                                trimming_threshold=0.01,
                                n_folds=3)
    else:
        model = dml.DoubleMLIRM(data_dml_base,
                                ml_g=regressors[name],
                                ml_m=classifiers[name],
                                trimming_threshold=0.01,
                                n_folds=3)
        if name in irm_nuisance_params:
            for nuisance, params in irm_nuisance_params[name].items():
                model.set_ml_nuisance_params(nuisance, TREATMENT_ELIG, params)

    model.fit(store_predictions=True)

    coef = model.coef[0]
    se = model.se[0]
    ci = model.confint(level=0.95).values[0]

    irm_results.append({
        "Model": "IRM",
        "Learner": name,
        "Coefficient": round(coef, 2),
        "Std_Error": round(se, 2),
        "CI_Lower": round(ci[0], 2),
        "CI_Upper": round(ci[1], 2),
        "Estimand": "ATE",
    })
    print(f"IRM-{name}: coef={coef:,.2f}, SE={se:,.2f}, "
          f"95% CI=[{ci[0]:,.2f}, {ci[1]:,.2f}]")

irm_df = pd.DataFrame(irm_results)
print("\nIRM results:")
print(irm_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
y_pos = range(len(irm_df))
ax.barh(y_pos, irm_df["Coefficient"], xerr=[
    irm_df["Coefficient"] - irm_df["CI_Lower"],
    irm_df["CI_Upper"] - irm_df["Coefficient"]],
    color=COLOR_IRM, alpha=0.7, edgecolor="white", capsize=4)
ax.axvline(x=naive_elig, color=COLOR_NAIVE, linestyle="--", linewidth=1.5,
           label=f"Naive difference (${naive_elig:,.0f})")
ax.set_yticks(y_pos)
ax.set_yticklabels(irm_df["Learner"])
ax.set_xlabel("Estimated Effect on Net Financial Assets ($)")
ax.set_title("IRM Estimates: Effect of 401(k) Eligibility (ATE)\nAcross ML Learners")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

The IRM estimates range from \$7,924 to \$8,559, with a mean of \$8,213. These are remarkably close to the PLR estimates (\$8,730 mean), differing by only about \$500. This convergence from two fundamentally different estimation strategies — partialling-out (PLR) and doubly robust/AIPW (IRM) — provides strong evidence that the ATE is robustly estimated in the \$8,000–\$9,000 range.

## Model 3: Interactive IV Model (IIVM)

### Why participation is endogenous

Participation (`p401`) is endogenous — it reflects a household's *choice*, driven by unobservable factors that also affect savings. Suppose two households are both eligible. Household A is financially savvy and enrolls immediately. Household B lives paycheck to paycheck and never enrolls. Comparing their savings mixes the 401(k) effect with pre-existing differences in financial discipline.

### Instrumental variables: eligibility as a nudge

The IIVM uses eligibility (`e401`) as an **instrument** for participation. The idea: eligibility acts like a nudge — it doesn't force anyone to participate, but it opens the door.

### Four types of households

| Type | Behavior | Interpretation |
|------|----------|---------------|
| **Always-takers** | Participate whether eligible or not | Highly motivated savers |
| **Never-takers** | Never participate, even when eligible | Prefer to spend |
| **Compliers** | Participate *because* eligible; would not otherwise | The marginal households |
| **Defiers** | Would participate if *not* eligible | Assumed not to exist |

The IIVM identifies the **LATE** — the causal effect specifically on *compliers*:

$$\theta_{LATE} = \frac{E[Y \mid Z=1] - E[Y \mid Z=0]}{E[D \mid Z=1] - E[D \mid Z=0]}$$

The numerator captures how much savings change when eligibility is "switched on." The denominator captures how many additional households actually participate when eligible. The ratio tells us: for each additional household nudged into participation, how much did their savings increase?

In [ ]:
print("Estimand: Local Average Treatment Effect (LATE) of 401(k) participation")
print("Treatment: p401 (participation) — endogenous")
print("Instrument: e401 (eligibility) — conditionally exogenous")
print("LATE applies to 'compliers': households who participate because eligible.\n")

iivm_results = []
regressors, classifiers = get_learners()

# IIVM-specific nuisance parameter tuning (from DoubleML docs)
iivm_nuisance_params = {
    "Random Forest": {
        "ml_g0": {"max_depth": 6, "max_features": 4, "min_samples_leaf": 7},
        "ml_g1": {"max_depth": 6, "max_features": 3, "min_samples_leaf": 5},
        "ml_m": {"max_depth": 6, "max_features": 3, "min_samples_leaf": 6},
        "ml_r1": {"max_depth": 4, "max_features": 7, "min_samples_leaf": 6},
    },
    "Decision Tree": {
        "ml_g0": {"ccp_alpha": 0.0016, "min_samples_split": 74,
                  "min_samples_leaf": 24},
        "ml_g1": {"ccp_alpha": 0.0018, "min_samples_split": 70,
                  "min_samples_leaf": 23},
        "ml_m": {"ccp_alpha": 0.0028, "min_samples_split": 167,
                 "min_samples_leaf": 55},
        "ml_r1": {"ccp_alpha": 0.0576, "min_samples_split": 55,
                  "min_samples_leaf": 18},
    },
    "XGBoost": {
        "ml_g0": {"eta": 0.1, "n_estimators": 9},
        "ml_g1": {"eta": 0.1, "n_estimators": 33},
        "ml_m": {"eta": 0.1, "n_estimators": 12},
        "ml_r1": {"eta": 0.1, "n_estimators": 25},
    },
}

for name in regressors:
    np.random.seed(RANDOM_SEED)
    if name == "Lasso":
        lasso_reg = make_pipeline(StandardScaler(),
                                  LassoCV(cv=5, max_iter=20000))
        model = dml.DoubleMLIIVM(data_dml_iv_flex,
                                 ml_g=lasso_reg,
                                 ml_m=classifiers[name],
                                 ml_r=classifiers[name],
                                 subgroups={"always_takers": False,
                                            "never_takers": True},
                                 trimming_threshold=0.01,
                                 n_folds=3)
    else:
        model = dml.DoubleMLIIVM(data_dml_base_iv,
                                 ml_g=regressors[name],
                                 ml_m=classifiers[name],
                                 ml_r=classifiers[name],
                                 subgroups={"always_takers": False,
                                            "never_takers": True},
                                 trimming_threshold=0.01,
                                 n_folds=3)
        if name in iivm_nuisance_params:
            for nuisance, params in iivm_nuisance_params[name].items():
                model.set_ml_nuisance_params(nuisance, TREATMENT_PART, params)

    model.fit(store_predictions=True)

    coef = model.coef[0]
    se = model.se[0]
    ci = model.confint(level=0.95).values[0]

    iivm_results.append({
        "Model": "IIVM",
        "Learner": name,
        "Coefficient": round(coef, 2),
        "Std_Error": round(se, 2),
        "CI_Lower": round(ci[0], 2),
        "CI_Upper": round(ci[1], 2),
        "Estimand": "LATE",
    })
    print(f"IIVM-{name}: coef={coef:,.2f}, SE={se:,.2f}, "
          f"95% CI=[{ci[0]:,.2f}, {ci[1]:,.2f}]")

iivm_df = pd.DataFrame(iivm_results)
print("\nIIVM results:")
print(iivm_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
y_pos = range(len(iivm_df))
ax.barh(y_pos, iivm_df["Coefficient"], xerr=[
    iivm_df["Coefficient"] - iivm_df["CI_Lower"],
    iivm_df["CI_Upper"] - iivm_df["Coefficient"]],
    color=COLOR_IIVM, alpha=0.7, edgecolor="white", capsize=4)
ax.axvline(x=naive_part, color=COLOR_NAIVE, linestyle="--", linewidth=1.5,
           label=f"Naive difference (${naive_part:,.0f})")
ax.set_yticks(y_pos)
ax.set_yticklabels(iivm_df["Learner"])
ax.set_xlabel("Estimated Effect on Net Financial Assets ($)")
ax.set_title("IIVM Estimates: Effect of 401(k) Participation (LATE)\nAcross ML Learners")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

The IIVM estimates range from \$11,215 to \$12,281, with a mean of \$11,746. This is substantially larger than the ATE from PLR/IRM (\$8,200–\$8,700), which is expected. The LATE captures the effect on compliers — households at the margin of participation — who benefit more from 401(k) access because the program creates *new* savings rather than reshuffling existing ones.

## Grand comparison

Let us put all 12 DML estimates alongside the two naive benchmarks in a single figure.

In [ ]:
all_results = pd.concat([plr_df, irm_df, iivm_df], ignore_index=True)
print("All DML results:")
print(all_results.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

labels = []
coefficients = []
errors_lower = []
errors_upper = []
colors = []

# Add naive benchmarks
labels.append("Naive (eligibility)")
coefficients.append(naive_elig)
errors_lower.append(0)
errors_upper.append(0)
colors.append(COLOR_NAIVE)

labels.append("Naive (participation)")
coefficients.append(naive_part)
errors_lower.append(0)
errors_upper.append(0)
colors.append(COLOR_NAIVE)

# Spacer
labels.append("")
coefficients.append(0)
errors_lower.append(0)
errors_upper.append(0)
colors.append("white")

# PLR results
for _, row in plr_df.iterrows():
    labels.append(f"PLR - {row['Learner']}")
    coefficients.append(row["Coefficient"])
    errors_lower.append(row["Coefficient"] - row["CI_Lower"])
    errors_upper.append(row["CI_Upper"] - row["Coefficient"])
    colors.append(COLOR_PLR)

labels.append("")
coefficients.append(0)
errors_lower.append(0)
errors_upper.append(0)
colors.append("white")

# IRM results
for _, row in irm_df.iterrows():
    labels.append(f"IRM - {row['Learner']}")
    coefficients.append(row["Coefficient"])
    errors_lower.append(row["Coefficient"] - row["CI_Lower"])
    errors_upper.append(row["CI_Upper"] - row["Coefficient"])
    colors.append(COLOR_IRM)

labels.append("")
coefficients.append(0)
errors_lower.append(0)
errors_upper.append(0)
colors.append("white")

# IIVM results
for _, row in iivm_df.iterrows():
    labels.append(f"IIVM - {row['Learner']}")
    coefficients.append(row["Coefficient"])
    errors_lower.append(row["Coefficient"] - row["CI_Lower"])
    errors_upper.append(row["CI_Upper"] - row["Coefficient"])
    colors.append(COLOR_IIVM)

y_pos = range(len(labels))
ax.barh(y_pos, coefficients,
        xerr=[errors_lower, errors_upper],
        color=colors, alpha=0.7, edgecolor="white", capsize=3)

ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.set_xlabel("Estimated Effect on Net Financial Assets ($)")
ax.set_title("Double Machine Learning: 401(k) Pension Study\n"
             "Comparing PLR (ATE), IRM (ATE), and IIVM (LATE)")
ax.axvline(x=0, color=NEAR_BLACK, linewidth=0.5)

legend_elements = [
    Patch(facecolor=COLOR_NAIVE, alpha=0.7, label="Naive (no confounding control)"),
    Patch(facecolor=COLOR_PLR, alpha=0.7, label="PLR (ATE)"),
    Patch(facecolor=COLOR_IRM, alpha=0.7, label="IRM (ATE)"),
    Patch(facecolor=COLOR_IIVM, alpha=0.7, label="IIVM (LATE)"),
]
ax.legend(handles=legend_elements, frameon=False, loc="upper right")

plt.tight_layout()
plt.show()

The grand comparison tells a three-part story:

1. **Confounding bias is massive.** The naive estimates (gray, \$19,559 and \$27,372) are more than double the DML estimates.
2. **PLR and IRM agree.** Both estimate the ATE at roughly \$8,000–\$9,400 regardless of the ML learner or estimation approach.
3. **LATE > ATE.** The IIVM estimates (teal, ~\$11,746) are systematically higher because compliers — marginal participants — benefit more than the population average.

| Model | Estimand | Mean Estimate | Range Across Learners |
|-------|----------|---------------|----------------------|
| Naive (eligibility) | — | \$19,559 | — |
| PLR | ATE | \$8,730 | \$7,823 – \$9,371 |
| IRM | ATE | \$8,213 | \$7,924 – \$8,559 |
| IIVM | LATE | \$11,746 | \$11,215 – \$12,281 |
| Naive (participation) | — | \$27,372 | — |

### Why is the LATE larger than the ATE?

The ATE averages the effect across *all* households, including always-takers who would have saved in other vehicles anyway. For them, the 401(k) may simply *reshuffle* savings, producing a smaller net effect. Compliers, by contrast, are households whose savings behavior genuinely changes with 401(k) access — the program creates *new* savings rather than reshuffling existing ones.

For policymakers deciding whether to expand 401(k) eligibility, the LATE (~\$12,000) is arguably more relevant than the ATE (~\$8,500), because newly eligible households are compliers by definition.

## Summary

**Key findings:**

1. **Confounding bias is real and large.** The naive estimate overstates the eligibility effect by \$10,829 (124%). Income is the primary confounder.
2. **The true ATE is ~\$8,500.** PLR and IRM agree across four ML learners, with estimates ranging from \$7,823 to \$9,371.
3. **Compliers benefit more (~\$11,746).** The LATE from the IIVM is larger because marginal participants are the ones whose savings behavior genuinely changes.
4. **Results are robust.** Four ML learners (Lasso, Random Forest, Decision Trees, XGBoost) and two estimation frameworks (PLR, IRM) all converge on similar estimates.

**Limitations:**

- The analysis assumes eligibility is as good as randomly assigned after conditioning on observables
- Cross-sectional data (1991 SIPP) — dynamic effects are not captured
- Extreme asset values may influence mean-based ATE estimates

## Exercises

1. **Change the number of cross-fitting folds.** Re-run the PLR model with `n_folds=5` and `n_folds=10`. How do the estimates change? Does increased folding improve precision?

2. **Explore subgroup effects.** Split the data by marital status (`marr == 1` vs. `marr == 0`) and estimate the PLR model separately for each group. Is the ATE larger for married or unmarried households?

3. **Test instrument strength.** Run a first-stage regression of `p401` on `e401` controlling for the base covariates. What is the F-statistic? Does the instrument satisfy the rule-of-thumb F > 10?

## References

1. [Chernozhukov, V., et al. (2018). Double/Debiased Machine Learning for Treatment and Structural Parameters. *The Econometrics Journal*, 21(1), C1–C68.](https://doi.org/10.1111/ectj.12097)
2. [Poterba, J., Venti, S., and Wise, D. (1995). Do 401(k) contributions crowd out other personal saving? *Journal of Public Economics*, 58(1), 1–32.](https://doi.org/10.1016/0047-2727(94)01462-W)
3. [DoubleML — An Object-Oriented Implementation of Double Machine Learning in Python](https://docs.doubleml.org/stable/)
4. [1991 Survey of Income and Program Participation (SIPP) — U.S. Census Bureau](https://www.census.gov/programs-surveys/sipp.html)

Full blog post: [carlos-mendez.org/post/python_doubleml_pension/](https://carlos-mendez.org/post/python_doubleml_pension/)